# Para utilizarlo con collab

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
# raiz="drive/MyDrive/proyecto_mineria/"
raiz=""

# Cargar datos y generar los conjuntos de entrenamiento y test

In [2]:
import pandas as pd

# file_path = 'data_set_limpio.pkl'
file_path = f'{raiz}datasets_pkl/data_set_limpio_sin_not_for_sale.pkl'

df = pd.read_pickle(file_path)

# Display the loaded DataFrame
print(df.columns)
df.sample(n=5)


Index(['Nat', 'Division', 'Club', 'Based', 'Preferred Foot', 'Right Foot',
       'Left Foot', 'Position', 'Height', 'Weight', 'Age', 'Wage', 'AT Apps',
       'AT Gls', 'Team', 'Caps', 'Yth Apps', 'Style', 'Rc Injury', 'Best Role',
       'Best Duty', 'Best Pos', 'Acc', 'Aer', 'Agg', 'Agi', 'Ant', 'Bal',
       'Bra', 'Cmd', 'Com', 'Cmp', 'Cnt', 'Cor', 'Cro', 'Dec', 'Det', 'Dri',
       'Ecc', 'Fin', 'Fir', 'Fla', 'Fre', 'Han', 'Hea', 'Jum', 'Kic', 'Ldr',
       'Lon', 'L Th', 'Mar', 'Nat .1', 'OtB', '1v1', 'Pac', 'Pas', 'Pen',
       'Pos', 'Pun', 'Ref', 'TRO', 'Sta', 'Str', 'Tck', 'Tea', 'Tec', 'Thr',
       'Vis', 'Wor', 'transfer_value_estimado'],
      dtype='object')


,Nat,Division,Club,Based,Preferred Foot,Right Foot,Left Foot,Position,Height,Weight,...,TRO,Sta,Str,Tck,Tea,Tec,Thr,Vis,Wor,transfer_value_estimado
19154,COL,-,-,Colombia,Right,Very Strong,Weak,AM (RC),170,68,...,3,12,10,3,8,13,2,10,6,0
117567,BRA,Swedish Premier Division,IFK Värnamo,Sweden (Premier Division),Left Only,Weak,Very Strong,D (C),192,82,...,3,12,14,12,10,11,3,10,10,83000
92505,HUN,Hungarian Division III - West,Kelen SC,Hungary (Division III - West),Right,Very Strong,Reasonable,"AM (RL), ST (C)",179,74,...,3,8,10,6,8,10,2,9,8,60000
176380,ESP,-,-,Spain,Left,Weak,Very Strong,GK,182,81,...,10,4,6,3,8,5,10,5,7,0
79022,SRB,Serbian SuperLeague,Mladost,Serbia (SuperLeague),Right,Very Strong,Weak,D/WB (R),180,68,...,1,11,11,12,10,12,4,10,10,412500


In [3]:
import pandas as pd
visualizar_todo=True
if visualizar_todo:
    # Show all rows
    pd.set_option('display.max_rows', None)

    # Show all columns
    pd.set_option('display.max_columns', None)

    # Show full content of each cell (prevent text truncation)
    pd.set_option('display.max_colwidth', None)


## Dataframe solo con features numericas

In [4]:
import numpy as np
# Create new dataframe with only numeric columns
only_numeric_df = df.select_dtypes(include=[np.number])
print(only_numeric_df.columns)
# # Optional: Create a separate dataframe for non-numeric columns
# only_categoric_df = df.select_dtypes(exclude=[np.number])

Index(['Height', 'Weight', 'Age', 'Wage', 'AT Apps', 'AT Gls', 'Caps',
       'Yth Apps', 'Acc', 'Aer', 'Agg', 'Agi', 'Ant', 'Bal', 'Bra', 'Cmd',
       'Com', 'Cmp', 'Cnt', 'Cor', 'Cro', 'Dec', 'Det', 'Dri', 'Ecc', 'Fin',
       'Fir', 'Fla', 'Fre', 'Han', 'Hea', 'Jum', 'Kic', 'Ldr', 'Lon', 'L Th',
       'Mar', 'Nat .1', 'OtB', '1v1', 'Pac', 'Pas', 'Pen', 'Pos', 'Pun', 'Ref',
       'TRO', 'Sta', 'Str', 'Tck', 'Tea', 'Tec', 'Thr', 'Vis', 'Wor',
       'transfer_value_estimado'],
      dtype='object')


# Generar conjuntos de entrenamiento y test

In [5]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [6]:
import clustering as cl

kmeans_model_club = cl.fit_kmeans(train_df, columna="Club")
kmeans_model_nat = cl.fit_kmeans(train_df, columna="Nat")
kmeans_model_division = cl.fit_kmeans(train_df, columna="Division")


train_df = cl.apply_kmeans(train_df, kmeans_model_club)
train_df = cl.apply_kmeans(train_df, kmeans_model_nat)
train_df = cl.apply_kmeans(train_df, kmeans_model_division)

test_df = cl.apply_kmeans(test_df, kmeans_model_club)
test_df = cl.apply_kmeans(test_df, kmeans_model_nat)
test_df = cl.apply_kmeans(test_df, kmeans_model_division)



In [7]:
target="transfer_value_estimado"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

## One Hot para las features categóricas

In [8]:
categorical_cols=["Nat_cluster","Division_cluster","Club_cluster","Preferred Foot","Right Foot","Left Foot","Best Pos","Best Duty","Style","Best Role"]
X_train = pd.get_dummies(X_train, columns=categorical_cols)
X_test  = pd.get_dummies(X_test, columns=categorical_cols)

X_train, X_test = X_train.align(X_test, join='left', axis=1,fill_value=0)

In [9]:
X_train = X_train.select_dtypes(include=[np.number,np.bool])
X_test = X_test.select_dtypes(include=[np.number,np.bool])

# Función para registrar en un csv resultados de diferentes regresiones

In [10]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def log_results(model, method_name, X_train, y_train, X_test, y_test, filepath=f"{raiz}test_results_regression.csv"):
    # parameters
    params_dict = model.get_params()
    
    # Convert non-serializable objects
    for k, v in params_dict.items():
        try:
            json.dumps(v)
        except TypeError:
            params_dict[k] = str(v)

    params = json.dumps(params_dict)
    # Predictions
    y_pred = model.predict(X_test)
    
    # Metrics
    r2_train = model.score(X_train, y_train)
    r2_test = model.score(X_test, y_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # Row to store
    row = {
        "method": method_name,
        "hyperparameters": json.dumps(params),
        "r2_train": r2_train,
        "r2_test": r2_test,
        "mae": mae,
        "rmse": rmse
    }
    
    # Append or create CSV
    try:
        df = pd.read_csv(filepath)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    except FileNotFoundError:
        df = pd.DataFrame([row])
    
    df.to_csv(filepath, index=False)

In [11]:
X_train.shape

(149436, 160)

# REGRESIONES

In [12]:
RS=0

## LinearRegression

In [13]:
from sklearn.linear_model import LinearRegression
linear_reg = LinearRegression()
linear_reg.fit(X_train, y_train)

log_results(linear_reg,"LinearRegression",X_train,y_train,X_test,y_test)


## Ridge

In [14]:
from sklearn.linear_model import RidgeCV
ridge_reg = RidgeCV()
ridge_reg.fit(X_train, y_train)

log_results(ridge_reg,"RidgeCV",X_train,y_train,X_test,y_test)


## Lasso

In [15]:
from sklearn.linear_model import LassoCV
lasso_reg = LassoCV(random_state=RS)
lasso_reg.fit(X_train, y_train)

log_results(lasso_reg,"LassoCV",X_train,y_train,X_test,y_test)


## ElasticNet

In [16]:
from sklearn.linear_model import ElasticNetCV
elastic_net_reg = ElasticNetCV(random_state=RS)
elastic_net_reg.fit(X_train, y_train)

log_results(elastic_net_reg,"ElasticNetCV",X_train,y_train,X_test,y_test)


## MLPRegressor

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# random_state=RS,learning_rate_init=0.5,learning_rate="adaptive", max_iter=1000, tol=1e-4,verbose=False
mlp_reg = MLPRegressor(random_state=RS,learning_rate_init=0.01,learning_rate="adaptive", max_iter=500, tol=1e-4,verbose=False)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

mlp_reg.fit(X_train, y_train)
log_results(mlp_reg,"MLPRegressor",X_train,y_train,X_test,y_test)